In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Classifiers and regressors
from sklearn.dummy import DummyRegressor


from sklearn.model_selection import cross_validate

from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import RFECV 
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold

#import shap
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_log_error, mean_absolute_percentage_error, mean_squared_error


from custom_transformers import HostSinceTransformer, BathroomExtractor, UKHostBinaryEncoder, HostResponseOrdinalEncoder

In [263]:
X_train = pd.read_csv("data/X_train.csv")
X_test = pd.read_csv("data/X_test.csv")
y_train = pd.read_csv("data/y_train.csv").squeeze()  # Convert to Series
y_test = pd.read_csv("data/y_test.csv").squeeze()

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1465, 28), (629, 28), (1465,), (629,))

In [264]:
# Load preprocessing
import joblib
preprocessing_pipeline = joblib.load('data/preprocessed/preprocessing_pipeline.pkl')

# Regression with Log(y)

Before training and testing any model, it is important to establish a baseline score. This allows us to verify that our models are being trained properly and provides a consistent reference point for evaluating model

### Baseline Model

In [265]:
y_train_log = np.log(y_train)
y_test_log = np.log(y_test)

pipe_dummy = make_pipeline(preprocessing_pipeline, DummyRegressor())

cv_dummy = cross_validate(pipe_dummy, X_train, y_train_log, cv=5, return_train_score=True)
cv_dummy_score = pd.DataFrame(cv_dummy)
cv_dummy_score

,fit_time,score_time,test_score,train_score
0,0.013312,0.006117,-0.004274,0.0
1,0.009447,0.004982,-0.002520,0.0
2,0.008289,0.004477,-0.015587,0.0
3,0.008421,0.004211,-0.000144,0.0
4,0.008482,0.004498,-0.000553,0.0


The mean train score across all 5 cross-validation folds is close to 0, as expected for a baseline model. This confirms that the DummyRegressor provides a suitable reference point for evaluating the performace of more complex models.

Let's use linear models next

### Linear Models

The Ridge model from scikit-learn is an ordinary least squares (OLS) regression with L2 regularization. We will again perform cross-validation with 5 folds to evaluate its performance

In [266]:
pipe_ridge = make_pipeline(preprocessing_pipeline, Ridge())
cv_ridge = cross_validate(pipe_ridge, X_train, y_train_log, cv=5, return_train_score=True)
cv_ridge_score = pd.DataFrame(cv_ridge)
cv_ridge_score

,fit_time,score_time,test_score,train_score
0,0.009864,0.004639,0.611122,0.654380
1,0.008711,0.004373,0.642066,0.647003
2,0.008723,0.004508,0.538829,0.642046
3,0.008208,0.004475,0.672445,0.640478
4,0.008505,0.004390,0.582789,0.663348


The test scores have improved significantly compared to the DummyRegressor, now averaging around 0.61. However, the train score remains low, which may indicate underfitting. To address this, we can try hyperparameter optimization to see if performance improves.

In OLS with L2 regularization, the `alpha` hyperparameter controls the strength of regularization. Adjusting `alpha` helps manage the bias-variance tradeoff


In [267]:
param_grid = {
    "ridge__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}
gs = GridSearchCV(pipe_ridge, param_grid = param_grid, n_jobs=1, return_train_score=True)
gs.fit(X_train, y_train_log)
results = pd.DataFrame(gs.cv_results_)

results = results[["param_ridge__alpha", "mean_train_score", "mean_test_score"]]
results

,param_ridge__alpha,mean_train_score,mean_test_score
0,0.001,0.649540,0.584051
1,0.010,0.649540,0.584404
2,0.100,0.649537,0.587758
3,1.000,0.649451,0.609450
4,10.000,0.648778,0.634998
5,100.000,0.641303,0.628358
6,1000.000,0.577503,0.567497


Changing the `alpha` hyperparameter did not yield the expected improvements. The mean train score remains around 0.65, and smaller values of `alpha` do not lead to significant gains. This suggests that simply tuning regularization is not enough to improve model performance.

This limitation may be due to non linear relationships in the data that a linear model like Ridge cannot capture. Therefore, it is appropriate to try more expressive models that can learn non linear relationships in the data that a linear model like Ridge cannot capture, such as tree based models

### Tree-based Ensemble Models

Let's run four different tree-based ensemble models: RandomForestRegressor, LGBMRegressor, DecisionTreeRegressor, and XGBRegressor. For comparison, we will also include the DummyRegressor and ridge models as references

In [268]:
pipe_rf = make_pipeline(preprocessing_pipeline, RandomForestRegressor(random_state=123))

pipe_lgbm = make_pipeline(preprocessing_pipeline, LGBMRegressor(random_state=123))

pipe_dtr = make_pipeline(preprocessing_pipeline, DecisionTreeRegressor(random_state=123))

pipe_xgboost = make_pipeline(preprocessing_pipeline, XGBRegressor(random_state=123, verbosity=0))

regressors = {
    "dummy": pipe_dummy,
    "ridge": pipe_ridge,
    "random forest": pipe_rf,
    "LightGBM": pipe_lgbm,
    "decision tree": pipe_dtr,
    "xgboost": pipe_xgboost
}

In [269]:
summary_results = {}

for name, model in regressors.items():
    cv_result = cross_validate(model, X_train, y_train_log, cv=3, return_train_score=True)
    
    summary_results[name] = {
        "fit_time": np.mean(cv_result["fit_time"]),
        "score_time": np.mean(cv_result["score_time"]),
        "mean_test_score": np.mean(cv_result["test_score"]),
        "std_test_score": np.std(cv_result["test_score"]),
        "mean_train_score": np.mean(cv_result["train_score"]),
        "std_train_score": np.std(cv_result["train_score"])
    }
df_summary = pd.DataFrame(summary_results).T

/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor wa

In [270]:
df_summary

,fit_time,score_time,mean_test_score,std_test_score,mean_train_score,std_train_score
dummy,0.008546,0.004792,-0.004296,0.003037,0.000000,0.000000
ridge,0.007928,0.004617,0.605521,0.027808,0.650820,0.011603
random forest,0.450667,0.010895,0.754518,0.011562,0.963449,0.002408
LightGBM,0.157622,0.006982,0.771674,0.010422,0.972399,0.001808
decision tree,0.016013,0.005341,0.576455,0.030956,0.989937,0.011632
xgboost,0.082409,0.006352,0.749344,0.009868,0.995609,0.001197


Now, here we can see that all tree models except DecisionTreeRegressor performed better than the ridge model. Most tree-based models achieved at least a 0.05 improvement over ridge, with minimal difference in fit time. The best scoring model is LightGBM's LGBMRegressor, achieving a test score of 0.77 and a training score of 0.97. However, the large gap between train and test scores suggests possible overfitting. Let's perform hyperparameter optimization to see if we can improve the test scores and reduce overfitting.

### Hyperparameter Optimization for Best Model

### LightGBM

Overfitting in boosting models like LightGBM often happens when the model is not regularized enough, allowing it to fit noise in the training data.  
Common causes include:

- **Too many leaves or overly deep trees** (`num_leaves`, `max_depth`)
- **Too many boosting iterations** (`n_estimators`) without enough regularization
- **Weak regularization settings** (low `lambda_l1`, `lambda_l2`, or high `min_child_samples`)
- **High learning rate** (`learning_rate`), causing the model to fit too aggressively

With the information above I will play around by using sk-learn's RandomizedSearchCV to minimize the bias-variance tradeoff.

In [271]:
param_random_lightGB = {
    "lgbmregressor__n_estimators": np.arange(50, 250, 50),  # Number of boosting rounds
    "lgbmregressor__learning_rate": np.logspace(-4, -1, 5), # Step size
    "lgbmregressor__max_depth": np.arange(1, 5, 2), # Tree depth
    "lgbmregressor__num_leaves": np.arange(5, 10, 2)  # Number of leaves
}

rs_lightGBM = RandomizedSearchCV(pipe_lgbm, param_random_lightGB,  n_iter=100, cv=4, n_jobs=1, random_state=123, return_train_score=True, verbose=1)
rs_lightGBM.fit(X_train, y_train_log)

Fitting 4 folds for each of 100 candidates, totalling 400 fits


/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor wa

,estimator,Pipeline(step..._state=123))])
,param_distributions,"{'lgbmregressor__learning_rate': array([0.0001..., 0.1 ]), 'lgbmregressor__max_depth': array([1, 3]), 'lgbmregressor__n_estimators': array([ 50, 100, 150, 200]), 'lgbmregressor__num_leaves': array([5, 7, 9])}"
,n_iter,100
,scoring,None
,n_jobs,1
,refit,True
,cv,4
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,123
,error_score,nan


In [272]:
# Get the best score and hyperparameter values
rs_lightGBM.best_score_, rs_lightGBM.best_params_

(0.7775602639232391,
 {'lgbmregressor__num_leaves': 7,
  'lgbmregressor__n_estimators': 200,
  'lgbmregressor__max_depth': 3,
  'lgbmregressor__learning_rate': 0.1})

In [273]:
# Find the row for the best parameters
best_index = rs_lightGBM.best_index_

# Get the train score for that row
train_score_best = rs_lightGBM.cv_results_['mean_train_score'][best_index]

print("Best CV test score:", rs_lightGBM.best_score_)
print("Train score for best params:", train_score_best)

Best CV test score: 0.7775602639232391
Train score for best params: 0.8860599023907442


After rigorous testing and hyperparameter tuning, I identified a configuration that reduced the training score while slightly improving the test score.  
This resulted in a test–train gap of approximately **0.10**, which is within an acceptable range for boosting models like LightGBM

Now, I plan to experiment with **stacking ensemble methods** to explore whether model performance can be improved further

### Stacking

I will be stacking the best LGBMRegressor model, the ridge model and a KNNeighbourRegressor, forming a heterogeneous stack to better capture different biases of different models.

In [274]:
# Base estimators
estimators = [
    ("knn", KNeighborsRegressor()),
    ("lgb", LGBMRegressor(random_state=123, verbose=-1)),
    ("rg", Ridge(random_state=123))  # Ridge doesn't take random_state
]

# Stacking regressor
stacking_reg = StackingRegressor(
    estimators=estimators,
    final_estimator=RidgeCV(),
    passthrough=True,
    cv=5,
    n_jobs=-1
)

# Full pipeline
stacking_model = make_pipeline(
    preprocessing_pipeline,
    stacking_reg
)

# Parameter grid — names follow: pipeline_step__stack_step__estimator__param
param_dist = {
    # LightGBM tuning inside the stack
    "stackingregressor__lgb__n_estimators": np.arange(50, 250, 50),
    "stackingregressor__lgb__learning_rate": np.logspace(-4, -1, 5),
    "stackingregressor__lgb__max_depth": np.arange(1, 5, 2),
    "stackingregressor__lgb__num_leaves": np.arange(5, 10, 2),

    # RidgeCV final estimator
    "stackingregressor__final_estimator__alphas": [np.logspace(-3, 3, 7)],

    # kNN tuning
    "stackingregressor__knn__n_neighbors": [3, 5, 7, 9],

    # Ridge base model tuning
    "stackingregressor__rg__alpha": [0.1, 1.0, 10.0]
}

# Randomized search
rs_stacking_model = RandomizedSearchCV(
    stacking_model,
    param_distributions=param_dist,
    n_iter=50,
    cv=5,
    n_jobs=-1,
    random_state=123,
    return_train_score=True
)

rs_stacking_model.fit(X_train, y_train_log)

/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor wa

,estimator,Pipeline(step...rough=True))])
,param_distributions,"{'stackingregressor__final_estimator__alphas': [array([1.e-03...e+02, 1.e+03])], 'stackingregressor__knn__n_neighbors': [3, 5, ...], 'stackingregressor__lgb__learning_rate': array([0.0001..., 0.1 ]), 'stackingregressor__lgb__max_depth': array([1, 3]), ...}"
,n_iter,50
,scoring,None
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,123
,error_score,nan


In [275]:
# Find the row for the best parameters
best_index = rs_stacking_model.best_index_

# Get the train score for that row
train_score_best_stacking = rs_stacking_model.cv_results_['mean_train_score'][best_index]

print("Best CV test score:", rs_stacking_model.best_score_)
print("Train score for best params:", train_score_best_stacking)

Best CV test score: 0.7658825808579616
Train score for best params: 0.8734446276459898


### Scoring

In [276]:
# Best model so far
best_model_regression = rs_lightGBM.best_estimator_
y_pred_log = best_model_regression.predict(X_test)

/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [277]:
r2_log = r2_score(y_test_log, y_pred_log)
mae_log = mean_absolute_error(y_test_log, y_pred_log)
mse_log = mean_squared_error(y_test_log, y_pred_log)
mape_log = mean_absolute_percentage_error(y_test_log, y_pred_log)

print(f"R^2 on log scale: {r2_log:.3f}")
print(f"MAE on log scale: ${mae_log:.2f}")
print(f"MSE on log scale: {mse_log:.4f}")
print(f"MAPE on log scale: {mape_log:.2f}")


R^2 on log scale: 0.728
MAE on log scale: $0.22
MSE on log scale: 0.0900
MAPE on log scale: 0.05


#### Explanation of Regression Scores on the Log Scale

- **R² on log scale (0.727):**  
  The model explains **72.7%** of the variance in log-transformed prices.  
  In other words, if the true log(price) varies significantly, the model can capture most of those variations.  
  This is a solid score for a pricing model.

- **MAE on log scale (0.22):**  
  The average absolute difference between predicted and actual log(price) is **0.22**.  
  In the original price scale, predictions are typically within **±25%** of the true value.  
  For example, if the actual price is $100, the model will usually predict between **$80** and **$125**.  
  For pricing tasks, this level of error is reasonable.

- **MSE on log scale (0.09):**  
  The mean squared error in log space measures the average of the squared differences between predicted and actual log-transformed prices.  
  A value of **0.09** indicates small squared deviations in log units, meaning predictions are generally close to actual values.  
  This is a low error value, showing that large multiplicative mistakes are rare.

- **MAPE on log scale (0.05):**  
  The mean absolute percentage error is **0.05**, meaning the average proportional error in log(price) predictions is **5%**.  
  If the true price is $200, the average error is about $10.  
  This is an **excellent** result for a regression task.

**Note:**  
All scores above are calculated on log-transformed prices, so errors represent **proportional differences** rather than absolute dollar amounts. To interpret in terms of actual price, exponentiate the error values.  
For example, a log MAE of **0.22** means the predicted price is typically between **80%** and **125%** of the actual price.

In [278]:
# Convert back to original scale
y_pred_real = np.exp(y_pred_log)
y_test_real = np.exp(y_test_log)  # If y_test was also log-transformed

In [279]:


r2 = r2_score(y_test_real, y_pred_real)
mae = mean_absolute_error(y_test_real, y_pred_real)
msle = mean_squared_log_error(y_test_real, y_pred_real)
mape = mean_absolute_percentage_error(y_test_real, y_pred_real)

print(f"MSLE: {msle:.4f}")
print(f"R^2 on original scale: {r2:.3f}")
print(f"MAE on original price scale: ${mae:.2f}")
print(f"MAPE on original scale: {mape:.2f}")


MSLE: 0.0876
R^2 on original scale: 0.651
MAE on original price scale: $22.37
MAPE on original scale: 0.22


- **R² on original scale (0.651)**  
  The model explains **65.1%** of the variance in actual prices.  
  This means roughly two-thirds of price variation is captured by the features, with the remaining third due to noise or missing factors.  
  For real-world pricing tasks, an R² above 0.65 is solid, though there’s still room for improvement.  

- **MAE on original scale ($22.37)**  
  On average, predictions are off by about **$22.37** from the true price.  
  For example, if the actual price is $150, the model typically predicts between **$127.63** and **$172.37**.  
  This error is moderate if prices span a wide range (e.g., $50–$300), but relatively large if prices cluster in a narrower range (e.g., $100–$150).  

- **MSLE on original scale (0.0876)**  
  The mean squared logarithmic error measures squared differences in the log of predicted vs. actual prices.  
  A score of **0.0876** is low, indicating the model rarely makes large proportional errors.  

- **MAPE on original scale (0.22)**  
  The mean absolute percentage error of **22%** means that, on average, predictions differ from the true price by ±22%.  
  For example, if the true price is $200, the typical error would be about $44.  
  In pricing applications, a MAPE below 20% is often considered strong; 22% is acceptable but suggests there’s room for further tuning.  


# Interquartile Binning

Let's use interquartile binning instead of equal width binning. This way we can avoid issues like class imbalance, which can greatly affect many models' predictability.

In [280]:
# Bin into 5 equal-width intervals
y_train_bin = pd.qcut(y_train, q=8, labels=False)
y_test_bin = pd.qcut(y_test, q=8, labels=False)

### Baseline

In [281]:
from sklearn.dummy import DummyClassifier

pipe_dummyc = make_pipeline(preprocessing_pipeline, DummyClassifier(strategy='most_frequent'))
cv_dummy_classification = cross_validate(pipe_dummyc, X_train, y_train_bin, cv=5, return_train_score=True)
cv_dummy_classification_score = pd.DataFrame(cv_dummy_classification)
cv_dummy_classification_score

,fit_time,score_time,test_score,train_score
0,0.009979,0.004698,0.129693,0.129693
1,0.008192,0.004389,0.129693,0.129693
2,0.008549,0.005197,0.129693,0.129693
3,0.008828,0.004544,0.129693,0.129693
4,0.008448,0.004500,0.129693,0.129693


### Logistic Regression

In [282]:
pipe_logistic_regression = make_pipeline(preprocessing_pipeline, LogisticRegression(max_iter=50000, solver="lbfgs", penalty='l2', random_state=123))
cv_logistic_regression = cross_validate(pipe_logistic_regression, X_train, y_train_bin, cv=5, return_train_score=True)
cv_logistic_regression_score = pd.DataFrame(cv_logistic_regression)
cv_logistic_regression_score

/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 14156 iteration(s) (status=1):
STOP: TOTAL NO. OF F,G EVALUATIONS EXCEEDS LIMIT

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 14168 iteration(s) (status=1):
STOP: TOTAL NO. OF F,G EVALUATIONS EXCEEDS LIMIT

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linea

,fit_time,score_time,test_score,train_score
0,3.310892,0.006054,0.358362,0.437713
1,3.298546,0.005846,0.365188,0.438567
2,3.312866,0.005710,0.406143,0.421502
3,3.274792,0.005923,0.354949,0.418942
4,3.249624,0.005665,0.341297,0.421502


Here, both the train and test scores are low, which suggests that the logistic regression model is underfitting. To address this, I relaxed the regularization by testing much larger values of C and also increased the max_iter parameter to help the model converge.

In [ ]:
param_grid = {
    "logisticregression__C": [.001, 0.01, .1, 1, 10, 100, 1000, 10000, 100000, 1000000]
}
gs = GridSearchCV(pipe_logistic_regression, param_grid = param_grid, n_jobs=-1, return_train_score=True)
gs.fit(X_train, y_train_bin)
results = pd.DataFrame(gs.cv_results_)
results_summary = results[["param_logisticregression__C", "mean_test_score", "std_test_score", "mean_train_score", "std_train_score"]]
pd.DataFrame(results_summary)


/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 14137 iteration(s) (status=1):
STOP: TOTAL NO. OF F,G EVALUATIONS EXCEEDS LIMIT

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 14098 iteration(s) (status=1):
STOP: TOTAL NO. OF F,G EVALUATIONS EXCEEDS LIMIT

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linea

,param_logisticregression__C,mean_test_score,std_test_score,mean_train_score,std_train_score
0,0.001,0.277816,0.022598,0.303413,0.011224
1,0.010,0.324232,0.014318,0.378498,0.005358
2,0.100,0.352901,0.014576,0.413652,0.001365
3,1.000,0.365188,0.021907,0.427645,0.008624
4,10.000,0.363823,0.025043,0.427816,0.008728
5,100.000,0.369966,0.025776,0.422355,0.006675
6,1000.000,0.368601,0.024133,0.424232,0.010553
7,10000.000,0.365188,0.017403,0.427304,0.008758
8,100000.000,0.366553,0.020318,0.429010,0.010259
9,1000000.000,0.365188,0.025080,0.427304,0.007883


### SVC Model

In [284]:
from sklearn.svm import SVC


pipe_svc = make_pipeline(preprocessing_pipeline, SVC(kernel='rbf', probability=True, random_state=123))
cv_svc = cross_validate(pipe_svc, X_train, y_train_bin, cv=5, return_train_score=True)
cv_svc_score = pd.DataFrame(cv_svc)
cv_svc_score

,fit_time,score_time,test_score,train_score
0,0.370524,0.026617,0.136519,0.159556
1,0.364347,0.026405,0.143345,0.160410
2,0.363560,0.026236,0.170648,0.148464
3,0.360386,0.026282,0.119454,0.162969
4,0.365833,0.025887,0.160410,0.152730


As we can see, the SVC with RBF kernel does not perform well even compared to LogisticRegression, lets move on to something more powerful

### Tree-based Ensemble Models

In [285]:
### Tree-based Ensemble Models
pipe_rfc = make_pipeline(
    preprocessing_pipeline,
    RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        min_samples_leaf=10,
        max_features='sqrt',
        random_state=123
    )
)

pipe_lgbmc = make_pipeline(preprocessing_pipeline, LGBMClassifier(random_state=123))

pipe_dtrc = make_pipeline(
    preprocessing_pipeline,
    DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=10,
        max_features='sqrt',
        random_state=123
    )
)

pipe_xgboostc = make_pipeline(
    preprocessing_pipeline,
    XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='mlogloss',
        verbosity=0,
        random_state=123
    )
)


classifiers = {
    "dummy": pipe_dummyc,
    "random forest": pipe_rfc,
    "LightGBM": pipe_lgbmc,
    "decision tree": pipe_dtrc,
    "xgboost": pipe_xgboostc
}

# Use multiple scorers
scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'roc_auc_ovr']

summary_results = {}

for name, model in classifiers.items():
    cv_result = cross_validate(
        model,
        X_train,
        y_train_bin,
        scoring=scoring,
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=123),
        return_train_score=True
    )

    summary_results[name] = {
        "fit_time": np.mean(cv_result["fit_time"]),
        "score_time": np.mean(cv_result["score_time"]),
        "mean_train_accuracy": np.mean(cv_result["train_accuracy"]),
        "mean_test_accuracy": np.mean(cv_result["test_accuracy"]),
        "mean_precision": np.mean(cv_result["test_precision_macro"]),
        "mean_recall": np.mean(cv_result["test_recall_macro"]),
        "mean_f1": np.mean(cv_result["test_f1_macro"]),
        "mean_auc": np.mean(cv_result["test_roc_auc_ovr"]),
    }

df_summary = pd.DataFrame(summary_results).T
df_summary

/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

,fit_time,score_time,mean_train_accuracy,mean_test_accuracy,mean_precision,mean_recall,mean_f1,mean_auc
dummy,0.008399,0.014891,0.129693,0.129692,0.016212,0.125000,0.028701,0.500000
random forest,0.095658,0.025087,0.548114,0.393167,0.377682,0.392651,0.376445,0.830132
LightGBM,1.292391,0.028450,0.999317,0.425922,0.422728,0.424679,0.421746,0.816643
decision tree,0.010041,0.016560,0.405802,0.335155,0.339937,0.334472,0.312095,0.760426
xgboost,0.736567,0.021805,0.896249,0.427283,0.421808,0.426441,0.421391,0.831925


Without hyperparameter tuning, XGBoost achieves the highest test accuracy (0.427) among all models, but it also shows clear signs of overfitting, with a very high training accuracy (0.896) and a much lower test accuracy. This means the model fits the training data very well but fails to generalize to unseen data

While accuracy provides a quick baseline, it’s not the most reliable metric for this pricing classification task—especially in the presence of overfitting. In this context, precision is important because it helps correctly identify listings in the correct price bins and avoid misclassifying low-priced listings as high-priced ones. However, even though XGBoost has the highest precision (0.4218), its large train–test gap suggests that this result may not hold up on new data.

The F1 score, which balances precision and recall, also places XGBoost near the top (0.4214), but again the overfitting raises doubts about how well this performance will transfer to real-world predictions.

For ranking ability, the AUC (Area Under the Curve) score is 0.83, slightly ahead of the second-best model. This indicates XGBoost can rank listings reasonably well, but given the overfitting, this metric may be overly optimistic.

### Hyperparameter Optimization for Best Model (XGBoost)

Let us try and see if we can close the bias-variance gap by tuning. Like above, the model shows a clear sign of overfitting, which could be a result of:

- **Too many leaves or overly deep trees** (`num_leaves`, `max_depth`)
- **Too many boosting iterations** (`n_estimators`) without enough regularization
- **Weak regularization settings** (low `lambda_l1`, `lambda_l2`, or high `min_child_samples`)
- **High learning rate** (`learning_rate`), causing the model to fit too aggressively

In [287]:
# Hyperparameter space for XGBoost
param_random_xgb = {
    "xgbclassifier__n_estimators": np.arange(50, 250, 50),
    "xgbclassifier__max_depth": np.arange(3, 7, 2),
    "xgbclassifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "xgbclassifier__gamma": [1, 2, 3, 4, 5],
    "xgbclassifier__lambda": np.arange(8, 20, 2), # L2 regularization term
    "xgbclassifier__alpha": np.arange(6, 18, 2), # L1 regularization term
    "xgbclassifier__verbosity": [0]
}

# ---- pipeline ----
pipe_xgb = make_pipeline(
    preprocessing_pipeline,
    XGBClassifier(
        random_state=123,
        verbosity=0,
        eval_metric="logloss"
    )
)

# ---- RandomizedSearchCV ----
rs_xgb = RandomizedSearchCV(
    estimator=pipe_xgb,
    param_distributions=param_random_xgb,
    n_iter=100,
    cv=4,
    n_jobs=-1,
    random_state=123,
    scoring=scoring,        # your scoring dict
    refit="accuracy",       # pick your main metric
    return_train_score=True
)

# ---- fit ----
rs_xgb.fit(X_train, y_train_bin)

# ---- report ----
best_index  = rs_xgb.best_index_
best_test   = rs_xgb.best_score_
best_train = rs_xgb.cv_results_[f"mean_train_accuracy"][best_index]


print("Best CV test score:", best_test)
print("Train score for best params:", best_train)
print("Best params:", rs_xgb.best_params_)
print("Generalization gap (train - test):", best_train - best_test)


ValueError: 
All the 400 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
400 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/pipeline.py", line 663, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/xgboost/cor

# Conclusion

In [ ]:
import os, json
import pandas as pd
import joblib
# from sklearn.pipeline import Pipeline  # if you’re building it here

# 1) Load the exact raw DataFrame you trained on
X = pd.read_csv("data/listings.csv")

# 2) Build or load your FULL pipeline (preprocessor + regressor)
#    Example: final_model = Pipeline([("pre", preprocessing_pipeline), ("model", best_model_regression)])
final_model = best_model_regression  # <- must include the preprocessor step

# 3) Attach the raw input schema so app.py can read it
final_model.input_schema_ = list(X.columns)

# 4) Save
os.makedirs("model", exist_ok=True)
joblib.dump(final_model, "model/final_model.pkl")

# 5) (Optional) also save a JSON schema
schema = {
    "feature_order": list(X.columns),
    "dtypes": {c: str(X[c].dtype) for c in X.columns}
}
with open("model/raw_schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)
